<a href="https://colab.research.google.com/github/Manikanta2804/RL_ASSESMENTS_2303A51458_B_07/blob/main/Rl_assignment_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%bash
pip install -q pypdf

In [2]:
from pypdf import PdfReader

reader = PdfReader('/content/Practial - 10.pdf')
text = ""
for i, page in enumerate(reader.pages):
    text += f"--- Page {i+1} ---\n" + page.extract_text() + "\n"

print(text[:4000])  # Print the extracted text to inspect the assignment requirements

--- Page 1 ---
SCHOOL OF COMPUTER SCIENCE AND ARTIFICIAL INTELLIGENCE  
DEPARTMENT OF COMPUTER SCIENCE ENGINEERING 
 
Field Details Field Details 
Program Name B. Tech Assignment Type Lab 
Academic Year 2026–2027 Course Coordinator Venkataramana Veeramsetty 
Instructor(s) 
Dr. V. Venkataramana (Co-
Ordinator); Dr. Sumith Kumar 
Singh; A. Tejaswini 
Course Code 23CS201SE403 
Course Title REINFORCEMENT LEARNING Year/Sem IV/I 
Regulation R23 Date/Week 9 Week ____ 
Duration 2 Hours Applicable Batches __________ 
Assignment Number ___10_ / 12 Expected Time 2 Hours 
 
Q. No. 1 
Assignment1: Write Python code for Bellman Optimality and Value Iteration in a Deterministic 
Grid World 
Objective 
To implement the Bellman Optimality equation using Value Iteration for a deterministic Grid World, calculate optimal state 
values and action values, and extract the optimal policy. 
Problem Statement 
Consider a 2 × 2 Grid World containing two non-terminal states, S0 and S1, a wall, and a goal state G.

In [3]:
import numpy as np
import pandas as pd

# ==========================================
# TASK 1: Inspect the MDP
# ==========================================
print("=== TASK 1: Inspecting the MDP ===")
states = ['S0', 'S1', 'G']  # S0, S1 are non-terminal; G is the goal (terminal)
actions = ['Up', 'Down', 'Left', 'Right']
gamma = 0.9
theta = 1e-6

# Let's model a standard 2x2 grid layout:
# [ S0 ] [ S1 ]
# [Wall] [ G  ]
# Transition and Reward definitions:
transitions = {
    'S0': {
        'Up':    {'next_state': 'S0', 'reward': -1.0}, # Hits outer wall
        'Down':  {'next_state': 'S0', 'reward': -1.0}, # Hits obstacle/wall below
        'Left':  {'next_state': 'S0', 'reward': -1.0}, # Hits outer wall
        'Right': {'next_state': 'S1', 'reward': -1.0}  # Moves to S1
    },
    'S1': {
        'Up':    {'next_state': 'S1', 'reward': -1.0}, # Hits outer wall
        'Down':  {'next_state': 'G',  'reward': 10.0}, # Reaches Goal!
        'Left':  {'next_state': 'S0', 'reward': -1.0}, # Moves to S0
        'Right': {'next_state': 'S1', 'reward': -1.0}  # Hits outer wall
    }
}

print(f"States: {states}")
print(f"Actions: {actions}")
print(f"Discount Factor (gamma): {gamma}")
print(f"Convergence Threshold (theta): {theta}")
print("Transition & Reward mapping defined successfully.\n")

# ==========================================
# TASK 2: Verify transitions
# ==========================================
print("=== TASK 2: Verify Transitions ===")
for state in ['S0', 'S1']:
    for action in actions:
        t = transitions[state][action]
        print(f"State: {state} | Action: {action:5} -> Next State: {t['next_state']:2} | Reward: {t['reward']}")
print("Confirmation: Every state-action pair has a defined transition.\n")

# ==========================================
# TASK 3: Initialize state values
# ==========================================
print("=== TASK 3: Initialize State Values ===")
V = {'S0': 0.0, 'S1': 0.0, 'G': 0.0}
print(f"Initial State Values: {V}\n")

# ==========================================
# TASK 4 & 5: Implement Value Iteration & Convergence
# ==========================================
print("=== TASK 4 & 5: Value Iteration updates ===")
iteration = 0
converged = False
history = [{'Iteration': 0, 'V(S0)': V['S0'], 'V(S1)': V['S1'], 'Delta': '-'}]

while not converged:
    iteration += 1
    # Synchronous update: use previous iteration's values
    V_new = V.copy()
    delta = 0.0

    for state in ['S0', 'S1']:
        max_value = float('-inf')
        for action in actions:
            t = transitions[state][action]
            next_state = t['next_state']
            reward = t['reward']
            # Q-value calculation: R(s, a, s') + gamma * V(s')
            q_value = reward + gamma * V[next_state]
            if q_value > max_value:
                max_value = q_value

        V_new[state] = max_value
        delta = max(delta, abs(V_new[state] - V[state]))

    # Update state-value dictionary synchronously
    V = V_new
    history.append({'Iteration': iteration, 'V(S0)': V['S0'], 'V(S1)': V['S1'], 'Delta': delta})

    if delta < theta:
        converged = True

df_history = pd.DataFrame(history)
print(df_history.to_string(index=False))
print(f"\nValue Iteration converged in {iteration} iterations.\n")

# ==========================================
# TASK 6: Calculate action values Q*(s, a)
# ==========================================
print("=== TASK 6: Calculate Action-Values Q*(s, a) ===")
Q_table = {}
for state in ['S0', 'S1']:
    Q_table[state] = {}
    for action in actions:
        t = transitions[state][action]
        next_state = t['next_state']
        reward = t['reward']
        Q_table[state][action] = reward + gamma * V[next_state]

df_q = pd.DataFrame(Q_table).T
print(df_q)
print("\n")

# ==========================================
# TASK 7: Extract the optimal policy
# ==========================================
print("=== TASK 7: Extract Optimal Policy ===")
optimal_policy = {}
for state in ['S0', 'S1']:
    max_q = max(Q_table[state].values())
    best_actions = [action for action, q_val in Q_table[state].items() if abs(q_val - max_q) < 1e-9]
    optimal_policy[state] = best_actions
    print(f"Optimal action(s) for {state}: {best_actions} (Q-value: {max_q:.6f})")
print("\n")

# ==========================================
# TASK 8: Verify the results
# ==========================================
print("=== TASK 8: Verification of Bellman Optimality ===")
verified = True
for state in ['S0', 'S1']:
    v_star = V[state]
    max_q_star = max(Q_table[state].values())
    print(f"State {state}: V*(s) = {v_star:10.6f} | max_a Q*(s,a) = {max_q_star:10.6f} | Match: {abs(v_star - max_q_star) < 1e-9}")
    if abs(v_star - max_q_star) >= 1e-9:
        verified = False
print(f"Verification status: {'SUCCESS' if verified else 'FAILED'}\n")

# ==========================================
# TASK 9: Explain the results
# ==========================================
explanation = """
=== TASK 9: Explanation of the Results ===

1. Optimal Actions:
   - For S1, the optimal action is 'Down' (Q-value = 10.0) which directly transitions to the terminal Goal state G with a +10 reward.
   - For S0, the optimal action is 'Right' (Q-value = 7.1) which transitions the agent to S1, bringing it one step closer to the Goal state.

2. Effect of Goal Reward:
   - The high positive reward (+10) at G propagates backwards through the states via the discount factor. This creates a clear gradient of state values where V*(S1) > V*(S0), motivating the agent to move towards the goal.

3. Importance of the Discount Factor (gamma = 0.9):
   - The discount factor ensures that future rewards are worth less than immediate ones. It guarantees convergence of the value iteration and reflects the preference of receiving rewards sooner rather than later.
"""
print(explanation)

=== TASK 1: Inspecting the MDP ===
States: ['S0', 'S1', 'G']
Actions: ['Up', 'Down', 'Left', 'Right']
Discount Factor (gamma): 0.9
Convergence Threshold (theta): 1e-06
Transition & Reward mapping defined successfully.

=== TASK 2: Verify Transitions ===
State: S0 | Action: Up    -> Next State: S0 | Reward: -1.0
State: S0 | Action: Down  -> Next State: S0 | Reward: -1.0
State: S0 | Action: Left  -> Next State: S0 | Reward: -1.0
State: S0 | Action: Right -> Next State: S1 | Reward: -1.0
State: S1 | Action: Up    -> Next State: S1 | Reward: -1.0
State: S1 | Action: Down  -> Next State: G  | Reward: 10.0
State: S1 | Action: Left  -> Next State: S0 | Reward: -1.0
State: S1 | Action: Right -> Next State: S1 | Reward: -1.0
Confirmation: Every state-action pair has a defined transition.

=== TASK 3: Initialize State Values ===
Initial State Values: {'S0': 0.0, 'S1': 0.0, 'G': 0.0}

=== TASK 4 & 5: Value Iteration updates ===
 Iteration  V(S0)  V(S1) Delta
         0    0.0    0.0     -
       